<a href="https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maha-naveed77/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
#One row = one content page, belonging to one client, on one calendar day (fact_content_daily_performance, grain confirmed by Query 1: grouping by report_date + client_hash_id + content_hash_id returns zero duplicate rows). Time window: developing on the mid-panel month month = '2026-03', spanning 2026-03-01 to 2026-03-31, with 9,841,378 rows in that month alone. The final month (June 2026, fact_content_daily_performance_sample.parquet) is treated as a sealed test set only.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
#Features: gsc_impressions, gsc_clicks, gsc_sum_position, sessions_ai, scroll_events — all observed same-day signals.
#Label/proxy: a future decline/recovery outcome, to be built from a forward window beyond March — not yet defined, deliberately deferred to the modeling weeks.
#Context: client_hash_id, content_hash_id, gsc_data_available, ga4_data_available, plus dim_clients.gsc_data_start/ga4_data_start for join and coverage checks — used to understand the data, not as model inputs.
#Excluded: any pre-existing FlyRank score/priority/health-style column, if one exists in this table — excluded because it would be a product decision, not an observed signal, and would let a model copy an existing rule instead of learning anything new.— excluded because these are FlyRank's own rule outputs, not observable signals, and would let the model copy an existing decision rather than discover anything (the circular-result trap the lane guide warns about).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print(hf_token is not None, len(hf_token) if hf_token else 0)

True 37


In [7]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

In [8]:
BASE = "hf://datasets/FlyRank/internship-warehouse"

q_dim_clients = f"SELECT * FROM read_parquet('{BASE}/dim_clients.parquet') LIMIT 5"
print(con.execute(q_dim_clients).df())

            client_hash_id  is_active  has_gsc_access  has_ga4_access  \
0  client_04660893ae39614a       True            True            True   
1  client_05475c07ed21a83a       True           False           False   
2  client_06d356715a8ff3b6       True            True            True   
3  client_0797ff3a1fc9a6a5       True           False           False   
4  client_08a6a72ff48e62c0       True            True           False   

                  access_profile client_created_date client_updated_date  \
0                    gsc_and_ga4          2026-04-15          2026-06-27   
1  no_search_or_analytics_access          2026-04-01          2026-06-27   
2                    gsc_and_ga4          2026-03-23          2026-07-05   
3  no_search_or_analytics_access          2025-05-26          2026-06-27   
4                       gsc_only          2025-05-26          2026-06-27   

  gsc_data_start ga4_data_start  
0            NaT     2026-05-22  
1            NaT            NaT  
2 

In [9]:
q_dim_content = f"SELECT * FROM read_parquet('{BASE}/dim_content.parquet') LIMIT 5"
print(con.execute(q_dim_content).df())

            client_hash_id           content_hash_id  \
0  client_04660893ae39614a  content_004de9653278b5a4   
1  client_04660893ae39614a  content_00dc5efae381b2ab   
2  client_04660893ae39614a  content_01410f2556c327ac   
3  client_04660893ae39614a  content_019f27f634053ca7   
4  client_04660893ae39614a  content_01efa71faea45dcc   

            keyword_hash_id           url_hash_id  keyword_char_count  \
0  keyword_e754999ab88dd9f2  url_d6091f18cf628794                  22   
1  keyword_4329d7aede8e208b  url_3a66d2f2e36823ca                  31   
2  keyword_9b08047d3d2a0406  url_809eda7a7e20b3b2                  22   
3  keyword_e7cec7ab1804c1c2  url_5fb42bafc4399861                  14   
4  keyword_56b0062a1d8b7524  url_ece0abc3e5fb75f9                  24   

   keyword_token_count  url_char_count content_created_date  \
0                    4             108           2026-05-30   
1                    6              95           2026-06-12   
2                    5             

In [10]:
q_peek = f"""
SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
LIMIT 5
"""
print(con.execute(q_peek).df())

  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
3  2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
4  2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True            True                True               False   
2            True            True                True               False   
3            True            True                True               False   
4            True            True                True               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               30           0               115  ...     

In [11]:
q1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
"""
print(con.execute(q1).df())  # expect: EMPTY — confirms one row per (date, client, content)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []


In [12]:
q2 = f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-03'
"""
print(con.execute(q2).df())

    n_rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31


In [13]:
q3 = f"""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4
FROM read_parquet('{BASE}/fact_content_daily_performance/*/*.parquet')
WHERE month = '2026-03'
"""
print(con.execute(q3).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_gsc  rows_with_ga4
0     9841378        3611061         413966


In [ ]:
#Grain (Query 1): zero rows returned when grouping by (report_date, client_hash_id, content_hash_id) and filtering to counts > 1 — confirms the stated grain is correct, one row truly is one page-client-day.
#Row count and date span (Query 2): 9,841,378 rows, spanning 2026-03-01 through 2026-03-31 — a full calendar month, as intended.
#Availability (Query 3): of 9,841,378 total rows, only 3,611,061 (36.7%) have gsc_data_available IS TRUE, and just 413,966 (4.2%) have ga4_data_available IS TRUE. This is a much lower availability rate than I assumed going in — most rows in this table lack GA4 signal entirely, and even GSC data is missing for nearly two-thirds of rows.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
#This slice covers one month (March 2026) across all clients, but availability is highly uneven: only 36.7% of rows have GSC data and just 4.2% have GA4 data, so most engagement-based features (sessions_ai, scroll_events) will be null or zero for the vast majority of rows — any model built on GA4-derived features will only really be trained on a thin slice of the full data. Clients also have staggered gsc_data_start/ga4_data_start dates (confirmed earlier from dim_clients), so low availability isn't random — it reflects real onboarding gaps, not missing-at-random data, and any comparison across clients needs to account for that rather than treating a zero as a true zero.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.